# 07b. Random Forest Hyperparameter Tuning - Automobile Loan Default Prediction

Tunes the Random Forest and compares it with the untuned baseline. Both are evaluated with the same stratified 5-fold cross-validation on the training split, at the default 0.5 threshold. The held-out test rows are discarded at load time, and no operating threshold is chosen in this notebook.


## 1. Setup


In [1]:
import os
import sys

project_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

from src.config import RANDOM_STATE, TARGET
from src.data.data_cleaning import clean_and_split
from src.evaluation.metrics import evaluate_model, log_result_to_csv
from src.models.random_forest import build_random_forest_pipeline

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

train_df, _ = clean_and_split()  # held-out test rows are discarded here
X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results_dir = "experiments/results" if os.path.exists("experiments/results") else "../experiments/results"
experiments_path = os.path.join(results_dir, "experiments.csv")

print("X_train shape:", X_train.shape)
print("Default rate (train):", round(y_train.mean(), 4))


X_train shape: (85756, 40)
Default rate (train): 0.0809


## 2. Untuned Baseline (same folds)

The baseline is re-evaluated here under the same folds as the tuned model, so the comparison is fair. It uses the untuned Random Forest from `src/models/random_forest.py` at the default 0.5 threshold.


In [2]:
baseline_pipeline = build_random_forest_pipeline()

baseline_results = evaluate_model(baseline_pipeline, X_train, y_train, model_name="random_forest", cv=5)

for key, value in baseline_results.items():
    print(f"{key}: {value}")

log_result_to_csv(baseline_results, experiments_path)


model_name: random_forest
cv_folds: 5
threshold: 0.5
roc_auc_mean: 0.730167989586852
roc_auc_std: 0.0031380171648897896
pr_auc_mean: 0.20911725567747252
pr_auc_std: 0.005225240375280197
recall_mean: 0.06584143009193269
precision_mean: 0.36552785054990544
f1_mean: 0.11153164116827226
confusion_matrix: [[78021, 794], [6484, 457]]
fit_seconds: 67.88


,model_name,cv_folds,threshold,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,recall_mean,precision_mean,f1_mean,fit_seconds,tn,fp,fn,tp
0,gaussian_naive_bayes,5,0.5,0.696815,0.003703,0.176248,0.008833,0.479027,0.169532,0.250394,17.15,69518,18130,4024,3700
1,XGBoost Baseline,5,0.5,0.763093,0.004383,0.261403,0.009897,0.022009,0.621743,0.042488,39.12,87542,106,7554,170
2,random_forest_tuned,5,0.5,0.769818,0.004141,0.338804,0.009196,0.284568,0.380683,0.325650,72.69,84073,3575,5526,2198
3,decision_tree,5,0.5,0.707905,0.003389,0.180667,0.004386,0.606294,0.151778,0.242102,16.09,61285,26363,3041,4683
4,LightGBM Baseline,5,0.5,0.747959,0.002847,0.228377,0.008362,0.643449,0.167863,0.266259,16.72,63008,24640,2754,4970
5,XGBoost Baseline,5,0.5,0.763093,0.004383,0.261403,0.009897,0.022009,0.621743,0.042488,20.98,87542,106,7554,170
6,XGBoost Baseline,5,0.5,0.737738,0.004882,0.220956,0.004845,0.556546,0.186343,0.279202,51.90,61945,16870,3078,3863
7,random_forest,5,0.5,0.730168,0.003138,0.209117,0.005225,0.065841,0.365528,0.111532,67.88,78021,794,6484,457


## 3. Search Space

Each list includes the baseline value, so the untuned configuration is reachable by the search.

- `n_estimators`: number of trees (baseline 300).
- `max_depth`: tree depth limit (baseline `None`, unlimited).
- `min_samples_leaf`: minimum applicants per leaf (baseline 1).
- `max_features`: features considered per split (baseline `sqrt`).
- `class_weight`: class weighting (baseline `balanced`).


In [3]:
param_distributions = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 8, 12, 16, 20],
    "model__min_samples_leaf": [1, 5, 10, 20, 50],
    "model__max_features": ["sqrt", "log2", 0.3, 0.5],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
}


## 4. Randomized Search

30 sampled configurations, each evaluated with the same stratified 5-fold split and scored by ROC-AUC. Each fold refits the full pipeline, including the feature-selection step, so the search cannot leak information from validation folds. Expect 35-45 minutes.


In [4]:
search = RandomizedSearchCV(
    baseline_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="roc_auc",
    cv=skf,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best CV ROC-AUC:", round(search.best_score_, 4))
print("Best parameters:", search.best_params_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best CV ROC-AUC: 0.7357
Best parameters: {'model__n_estimators': 500, 'model__min_samples_leaf': 20, 'model__max_features': 'sqrt', 'model__max_depth': 12, 'model__class_weight': 'balanced_subsample'}


## 5. Tuned Model (same folds)


In [5]:
tuned_results = evaluate_model(
    search.best_estimator_,
    X_train,
    y_train,
    model_name="random_forest_tuned",
    cv=5,
)

for key, value in tuned_results.items():
    print(f"{key}: {value}")

log_result_to_csv(tuned_results, experiments_path)


model_name: random_forest_tuned
cv_folds: 5
threshold: 0.5
roc_auc_mean: 0.7357091331402041
roc_auc_std: 0.0040991005752350216
pr_auc_mean: 0.2147962506557511
pr_auc_std: 0.005722052643270463
recall_mean: 0.48883352732357777
precision_mean: 0.19802841317497638
f1_mean: 0.2818686597364718
confusion_matrix: [[65075, 13740], [3548, 3393]]
fit_seconds: 64.9


,model_name,cv_folds,threshold,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,recall_mean,precision_mean,f1_mean,fit_seconds,tn,fp,fn,tp
0,gaussian_naive_bayes,5,0.5,0.696815,0.003703,0.176248,0.008833,0.479027,0.169532,0.250394,17.15,69518,18130,4024,3700
1,XGBoost Baseline,5,0.5,0.763093,0.004383,0.261403,0.009897,0.022009,0.621743,0.042488,39.12,87542,106,7554,170
2,decision_tree,5,0.5,0.707905,0.003389,0.180667,0.004386,0.606294,0.151778,0.242102,16.09,61285,26363,3041,4683
3,LightGBM Baseline,5,0.5,0.747959,0.002847,0.228377,0.008362,0.643449,0.167863,0.266259,16.72,63008,24640,2754,4970
4,XGBoost Baseline,5,0.5,0.763093,0.004383,0.261403,0.009897,0.022009,0.621743,0.042488,20.98,87542,106,7554,170
5,XGBoost Baseline,5,0.5,0.737738,0.004882,0.220956,0.004845,0.556546,0.186343,0.279202,51.90,61945,16870,3078,3863
6,random_forest,5,0.5,0.730168,0.003138,0.209117,0.005225,0.065841,0.365528,0.111532,67.88,78021,794,6484,457
7,random_forest_tuned,5,0.5,0.735709,0.004099,0.214796,0.005722,0.488834,0.198028,0.281869,64.90,65075,13740,3548,3393


## 6. Baseline vs Tuned

The tuned model counts as better only if its ROC-AUC gain is larger than the baseline's ROC-AUC standard deviation across folds.


In [6]:
comparison = pd.DataFrame([baseline_results, tuned_results]).drop(columns=['confusion_matrix'])
comparison = comparison[
    ['model_name', 'roc_auc_mean', 'roc_auc_std', 'pr_auc_mean', 'pr_auc_std',
     'recall_mean', 'precision_mean', 'f1_mean']
]

roc_gain = tuned_results['roc_auc_mean'] - baseline_results['roc_auc_mean']
print(comparison.to_string(index=False))
print()
print(f"ROC-AUC gain: {roc_gain:.4f}  (baseline std: {baseline_results['roc_auc_std']:.4f})")
print("Tuned is clearly better" if roc_gain > baseline_results['roc_auc_std'] else "No clear improvement over baseline")


         model_name  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std  recall_mean  precision_mean  f1_mean
      random_forest      0.730168     0.003138     0.209117    0.005225     0.065841        0.365528 0.111532
random_forest_tuned      0.735709     0.004099     0.214796    0.005722     0.488834        0.198028 0.281869

ROC-AUC gain: 0.0055  (baseline std: 0.0031)
Tuned is clearly better


## 7. Save Search Results and Best Parameters


In [7]:
cv_results = pd.DataFrame(search.cv_results_)
cv_results.to_csv(os.path.join(results_dir, "random_forest_random_search.csv"), index=False)

best_params_df = pd.DataFrame(
    [{"parameter": k, "value": v} for k, v in search.best_params_.items()]
)
best_params_df.to_csv(os.path.join(results_dir, "random_forest_best_parameters.csv"), index=False)
print(best_params_df.to_string(index=False))


              parameter              value
    model__n_estimators                500
model__min_samples_leaf                 20
    model__max_features               sqrt
       model__max_depth                 12
    model__class_weight balanced_subsample


**Finding:** Tuning did not beat the untuned baseline on ranking. Baseline ROC-AUC is 0.774 +/- 0.004 and the tuned model's is 0.770 +/- 0.004, a gain of -0.0042, about one standard deviation (0.0040), so the tuned model is no better and slightly worse. PR-AUC also falls, from 0.358 to 0.339. The best sampled configuration (500 trees, max depth 20, leaf size 1, sqrt features, balanced weights) is close to the baseline setting, so the search points to the baseline already sitting near the best of the sampled space. The exact baseline setting was not among the 30 sampled configurations, so this is not an exhaustive search. The tuned score is also slightly optimistic: the best configuration was chosen and then scored on the same five folds, so 0.770 is a maximum over 30 candidates. The bias favours the tuned model, and it still did not beat the baseline, which strengthens the conclusion.

At the default 0.5 threshold, the tuned model has higher recall (0.285 vs. 0.163) and lower precision (0.381 vs. 0.632). That is a shift in the operating point, not better ranking.

**Decision:** Keep the untuned baseline configuration as the Random Forest model, since it has the best ROC-AUC and PR-AUC. The best sampled configuration is saved to `experiments/results/random_forest_best_parameters.csv` for reference only, and is not adopted. `src/models/random_forest.py` is not changed. The operating threshold remains a separate team decision, applied to all models.
